# Chapter 13: Deploying Models in Production
## 📌 Summary
Menyelamatkan, memuat, dan menyajikan model ML di lingkungan produksi. Chapter ini mencakup **serialization (joblib/pickle), production pipelines, versioning**, dan **best practices**.

## 🧠 Theoretical Deep-Dive

### 13.1 Model Serialization
- **pickle**: Python built-in, tapi bisa lambat untuk large numpy arrays
- **joblib**: lebih efisien untuk scikit-learn models (parallel compression)
- **Format**: `.pkl` atau `.joblib`

### 13.2 Production Pipeline
Selalu simpan **entire pipeline** (preprocessing + model), bukan hanya model. Ini memastikan:
1. Preprocessing konsisten antara training dan inference
2. Single `.predict()` call untuk input raw
3. No data leakage risk

### 13.3 Versioning & Reproducibility
- Simpan **scikit-learn version**, **Python version**, dan **training metadata**
- Gunakan `__version__` attribute
- Track experiment dengan MLflow (praktik terbaik)

### 13.4 Considerations untuk Production
- **Monitoring**: track distribution shift (data drift)
- **Latency**: batch vs real-time prediction
- **Scalability**: joblib.Parallel untuk batch inference
- **A/B testing**: compare models di production

## 💻 Code Reproduction

In [ ]:
import joblib
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
print("Train accuracy:", clf.score(X_train, y_train))
print("Test accuracy:", clf.score(X_test, y_test).round(4))

os.makedirs("models", exist_ok=True)

# joblib (recommended for scikit-learn)
joblib.dump(clf, "models/rf_model.joblib")
clf_loaded = joblib.load("models/rf_model.joblib")
print("\nLoaded model accuracy:", clf_loaded.score(X_test, y_test).round(4))

# pickle
with open("models/rf_model.pkl", "wb") as f:
    pickle.dump(clf, f)
with open("models/rf_model.pkl", "rb") as f:
    clf_pkl = pickle.load(f)
print("Pickle model accuracy:", clf_pkl.score(X_test, y_test).round(4))

# Compare file sizes
print(f"\njoblib size: {os.path.getsize("models/rf_model.joblib")/1024:.1f} KB")
print(f"pickle size: {os.path.getsize("models/rf_model.pkl")/1024:.1f} KB")

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import os

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Full production pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", GradientBoostingClassifier(n_estimators=100, random_state=42))
])

pipe.fit(X_train, y_train)
print("Pipeline test accuracy:", pipe.score(X_test, y_test).round(4))
print("Classification Report:")
print(classification_report(y_test, pipe.predict(X_test)))

os.makedirs("models", exist_ok=True)
joblib.dump(pipe, "models/production_pipeline.joblib")
print("Pipeline saved!")

# Simulate production inference
loaded_pipe = joblib.load("models/production_pipeline.joblib")
new_samples = X_test[:5]
predictions = loaded_pipe.predict(new_samples)
probabilities = loaded_pipe.predict_proba(new_samples)
print("\nPredictions:", predictions)
print("Confidence (max proba):", probabilities.max(axis=1).round(3))

In [ ]:
import joblib
import json
import datetime
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import os, sys

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

os.makedirs("models", exist_ok=True)

# Save with metadata
model_metadata = {
    "model_type": type(clf).__name__,
    "hyperparameters": clf.get_params(),
    "train_accuracy": clf.score(X_train, y_train),
    "test_accuracy": clf.score(X_test, y_test),
    "n_features": X_train.shape[1],
    "n_classes": len(clf.classes_),
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version,
    "timestamp": datetime.datetime.now().isoformat(),
    "dataset": "iris"
}

joblib.dump(clf, "models/rf_v1.joblib")
with open("models/rf_v1_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model saved with metadata!")
print(json.dumps(model_metadata, indent=2))

In [ ]:
import joblib
import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=10000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_parts = [
    ("scaler", StandardScaler()),
]
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([("scaler", StandardScaler()), ("clf", RandomForestClassifier(n_estimators=100, random_state=42))])
pipe.fit(X_train, y_train)

# Batch prediction simulation
batch_sizes = [1, 10, 100, 1000]
for batch_size in batch_sizes:
    batch = X_test[:batch_size]
    start = time.time()
    for _ in range(10):
        preds = pipe.predict(batch)
    elapsed = (time.time() - start) / 10
    print(f"Batch size={batch_size:5}: avg time={elapsed*1000:.3f}ms, throughput={batch_size/elapsed:.0f} samples/sec")

In [ ]:
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Simulate model versioning and A/B testing
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Model A: Simple
model_a = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=50, random_state=42))
])
model_a.fit(X_train, y_train)

# Model B: More complex  
model_b = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=2, random_state=42))
])
model_b.fit(X_train, y_train)

# A/B comparison
print("Model A accuracy:", model_a.score(X_test, y_test).round(4))
print("Model B accuracy:", model_b.score(X_test, y_test).round(4))

# Save both versions
import os
os.makedirs("models", exist_ok=True)
joblib.dump(model_a, "models/model_v1.joblib")
joblib.dump(model_b, "models/model_v2.joblib")
print("\nBoth model versions saved!")

# Feature importance comparison
print("\nTop 3 features (Model A):", np.argsort(model_a.named_steps["clf"].feature_importances_)[::-1][:3])
print("Top 3 features (Model B):", np.argsort(model_b.named_steps["clf"].feature_importances_)[::-1][:3])

## ✅ Chapter Summary
- **Selalu gunakan joblib** untuk scikit-learn models (lebih efisien dari pickle untuk numpy arrays)
- **Simpan seluruh Pipeline**, bukan hanya model → konsistensi preprocessing
- **Metadata JSON** penting untuk reproducibility: versi, hyperparameters, tanggal, metrics
- **Batch inference** jauh lebih efisien dari single-sample; batch di production jika memungkinkan
- **Model versioning**: simpan beberapa versi dengan naming scheme yang jelas
- Di production riil: gunakan **MLflow** untuk experiment tracking + model registry